In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/energy_usage_raw.csv")



In [2]:

print("Raw rows:", len(df))
print(df.isna().sum())


Raw rows: 231
timestamp       0
device_id       4
device_name     0
room_id         0
room_name       0
energy_kwh     17
dtype: int64


In [3]:
df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed")
df["energy_kwh"] = pd.to_numeric(df["energy_kwh"], errors="coerce")
df["device_id"] = pd.to_numeric(df["device_id"], errors="coerce")

df = df.drop_duplicates()
df = df.dropna(subset=["device_id", "timestamp"])
df["device_id"] = df["device_id"].astype(int)

df.loc[df["energy_kwh"] < 0, "energy_kwh"] = np.nan

device_medians = df.groupby("device_id")["energy_kwh"].transform("median")
df["energy_kwh"] = df["energy_kwh"].fillna(device_medians)

df = df.dropna(subset=["energy_kwh"])

print("\nCleaned rows:", len(df))


Cleaned rows: 222


In [4]:

device_totals = df.groupby("device_name")["energy_kwh"].apply(lambda x: np.sum(x.to_numpy()))
device_averages = df.groupby("device_name")["energy_kwh"].apply(lambda x: np.mean(x.to_numpy()))

device_summary = pd.DataFrame({
    "total_energy_kwh": device_totals.round(3),
    "avg_energy_kwh": device_averages.round(3)
}).sort_values("total_energy_kwh", ascending=False)

print("\nDevice-level summary:")
print(device_summary)


Device-level summary:
                  total_energy_kwh  avg_energy_kwh
device_name                                       
Air Conditioner             66.884           1.454
Refrigerator                 9.437           0.410
Desktop Computer             9.116           0.480
Microwave Oven               8.986           0.374
Smart TV                     6.922           0.301
Ceiling Fan                  2.942           0.140
Table Lamp                   1.934           0.081
Air Purifier                 1.840           0.102
Printer                      0.736           0.031


In [5]:

room_summary = df.groupby("room_name")["energy_kwh"].sum().round(3).sort_values(ascending=False)

print("\nRoom-level summary:")
print(room_summary)



Room-level summary:
room_name
Living Room       41.578
Master Bedroom    35.170
Kitchen           18.423
Study              9.852
Bedroom 2          3.774
Name: energy_kwh, dtype: float64


In [6]:

df["date"] = df["timestamp"].dt.date
daily_room_summary = df.groupby(["date", "room_name"])["energy_kwh"].sum().round(3).unstack().fillna(0)

print("\nDaily energy by room:")
print(daily_room_summary)



Daily energy by room:
room_name   Bedroom 2  Kitchen  Living Room  Master Bedroom  Study
date                                                              
2026-06-13      0.213    2.753        5.435           7.033  1.136
2026-06-14      0.587    2.687        7.477           5.324  1.141
2026-06-15      0.551    2.828        4.909           5.598  1.010
2026-06-16      0.743    2.642        7.093           3.892  1.975
2026-06-17      0.597    2.602        5.285           6.632  2.158
2026-06-18      0.462    2.694        6.641           4.621  1.434
2026-06-19      0.621    2.216        4.739           2.070  0.998


In [7]:

df.to_csv("/content/energy_usage_cleaned.csv", index=False)
device_summary.to_csv("/content/device_summary.csv")
room_summary.to_csv("/content/room_summary.csv")
